# Experiment 2: phase transition and criticality

**Question.** Back on the square lattice, two dials are turned:

- **2a, initial density.** Does the outcome of a random soup change abruptly at some starting density? Which densities die out, which survive?
- **2b, avalanches.** Let the grid settle, add **one** live cell, wait until it settles again, and measure how far the disturbance spread. Repeat thousands of times, each time from the new settled state. In a *self-organized critical* system (Bak, Tang & Wiesenfeld 1987) avalanche sizes follow a power law with no typical scale; Bak, Chen & Creutz (1989) argued that Life is such a system.

| Term | Meaning |
|---|---|
| avalanche size | number of cells that differ, at some point, from the undisturbed run |
| avalanche duration | generations until the disturbed run is back on a cycle |
| power law | $P(s) \propto s^{-\alpha}$: a straight line on log–log axes, no typical size |
| finite-size cut-off | avalanches cannot exceed the grid, so even a power law bends down at the far end; for criticality the cut-off should move out as the grid grows |

## Predictions (written before any run)

| # | Question | Prediction |
|---|---|---|
| 1 | Final density for starting densities 0.2–0.6 | roughly constant |
| 2 | Below which starting density do most soups die out? Lower or higher than 0.19 on the random network? | lower, about 0.1 |
| 3 | What happens at starting densities 0.8 and 0.9? | a still state |
| 4 | Starting density with the longest transients | 0.1 |
| 5 | Avalanche sizes on log–log axes: straight line or bending down? | straight line |
| 6 | Slope, roughly | −1 |
| 7 | From $L$ = 32 to 128, does the largest avalanche keep growing or level off? | levels off |
| 8 | Is Life self-organized critical? | yes |

In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from life.avalanches import OPEN, avalanche, open_border, settle
from life.metrics import detect_cycle
from life.rules import CONWAY
from life.stats import fit_power_law
from life.topology import Lattice

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FIGURES = ROOT / "docs" / "figures"
DATA = ROOT / "data" / "exp02"
FIGURES.mkdir(parents=True, exist_ok=True)
DATA.mkdir(parents=True, exist_ok=True)

TORUS = Lattice("torus")
MAX_STEPS = 20_000

## 2a. Initial density sweep

64 × 64 torus, 14 starting densities, 50 soups each, followed to their cycle (cap 20 000 generations).

In [ ]:
SIZE = 64
DENSITIES = [0.01, 0.02, 0.05, 0.08, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
RUNS = 50

sweep = []  # start density, seed, transient, period, final density; -1 marks a censored run
start = time.perf_counter()
for rho0 in DENSITIES:
    for seed in range(RUNS):
        soup = np.random.default_rng([int(rho0 * 1000), seed]).random((SIZE, SIZE)) < rho0
        cycle = detect_cycle(soup, CONWAY, TORUS, MAX_STEPS)
        if cycle is None:
            sweep.append((rho0, seed, -1, -1, -1.0))
        else:
            alive = np.unpackbits(np.frombuffer(cycle.attractor, dtype=np.uint8))[: SIZE * SIZE]
            sweep.append((rho0, seed, cycle.transient, cycle.period, alive.mean()))
print(f"{len(sweep)} runs in {time.perf_counter() - start:.1f} s")
sweep = np.array(sweep)
np.savetxt(
    DATA / "density_sweep.csv",
    sweep,
    delimiter=",",
    fmt="%.6g",
    header="start_density,seed,transient,period,final_density",
    comments="",
)

header = ["start", "died", "censored", "median final (survivors)", "median T", "max T"]
widths = [6, 6, 9, 25, 9, 7]
print(" ".join(f"{h:>{w}}" for h, w in zip(header, widths, strict=True)))
for rho0 in DENSITIES:
    rows = sweep[sweep[:, 0] == rho0]
    done = rows[rows[:, 2] >= 0]
    died = np.sum(done[:, 4] == 0)
    alive = done[done[:, 4] > 0]
    final = np.median(alive[:, 4]) if len(alive) else 0.0
    print(
        f"{rho0:>6} {died:>4}/{len(rows)} {np.sum(rows[:, 2] < 0):>9} {final:>25.4f} "
        f"{np.median(done[:, 2]):>9.0f} {done[:, 2].max():>7.0f}"
    )

In [ ]:
x = np.array(DENSITIES)
died = []
final = []
transient = []
for rho0 in DENSITIES:
    done = sweep[(sweep[:, 0] == rho0) & (sweep[:, 2] >= 0)]
    died.append(np.mean(done[:, 4] == 0))
    alive = done[done[:, 4] > 0]
    final.append(np.median(alive[:, 4]) if len(alive) else np.nan)
    transient.append(np.median(done[:, 2]))

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, y, title in zip(
    axes,
    [died, final, transient],
    ["fraction of soups that die out", "median final density (survivors)", "median transient"],
    strict=True,
):
    ax.plot(x, y, "o-", color="#2f7d4f")
    ax.set_xlabel("initial density")
    ax.set_title(title)
fig.tight_layout()
fig.savefig(FIGURES / "exp02_density.png", dpi=150)
plt.show()

## 2b. Avalanches with open boundaries

Grids of side $L$ = 32, 64, 128. The outer two rows and columns are cleared after every step, so gliders that reach the edge leave the system. Start from a settled soup, then 5000 times: pick a random dead cell in the interior, make it alive, run until the grid is back on a cycle, record size and duration, continue from the new state. The first 500 avalanches are treated as warm-up and left out of the fits.

In [ ]:
SIZES = [32, 64, 128]
N_AVALANCHES = 5000
WARMUP = 500

records = []  # L, index, size, duration, censored
for L in SIZES:
    mask = open_border((L, L))
    rng = np.random.default_rng(L)
    state = settle(rng.random((L, L)) < 0.35, CONWAY, OPEN, mask, MAX_STEPS)
    start = time.perf_counter()
    for i in range(N_AVALANCHES):
        dead = np.argwhere(~state & mask)
        cell = tuple(int(v) for v in dead[rng.integers(len(dead))])
        result = avalanche(state, cell, CONWAY, OPEN, mask, MAX_STEPS)
        censored = result.state is None
        records.append((L, i, result.size, result.duration, int(censored)))
        if censored:
            print(f"L = {L}: avalanche {i} hit the cap; stopping this size")
            break
        state = result.state
    print(
        f"L = {L:>3}: {i + 1} avalanches in {time.perf_counter() - start:.1f} s, "
        f"final live density {state.sum() / mask.sum():.3f}"
    )

aval = np.array(records)
np.savetxt(
    DATA / "avalanches.csv",
    aval,
    fmt="%d",
    delimiter=",",
    header="L,index,size,duration,censored",
    comments="",
)

In [ ]:
def describe(values, label):
    fit = fit_power_law(values)
    verdict = "power law" if fit.loglik_ratio > 0 else "exponential"
    sure = "significant" if fit.p_value < 0.1 else "not significant"
    print(
        f"  {label:>8}: alpha = {fit.alpha:.2f} ± {fit.alpha_err:.2f}, xmin = {fit.xmin:>3}, "
        f"tail n = {fit.n_tail:>4}, KS = {fit.ks:.3f}, LR = {fit.loglik_ratio:+8.1f} "
        f"(p = {fit.p_value:.3f}) -> {verdict}, {sure}"
    )
    return fit


fits = {}
for L in SIZES:
    rows = aval[(aval[:, 0] == L) & (aval[:, 1] >= WARMUP) & (aval[:, 4] == 0)]
    interior = (L - 4) ** 2
    print(
        f"L = {L}: {len(rows)} avalanches, median size {np.median(rows[:, 2]):.0f}, "
        f"max size {rows[:, 2].max()} ({rows[:, 2].max() / interior:.0%} of the interior), "
        f"max duration {rows[:, 3].max()}"
    )
    fits[L] = (describe(rows[:, 2], "size"), describe(rows[:, 3], "duration"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = {32: "#3b6ea8", 64: "#2f7d4f", 128: "#d1495b"}
for ax, col, label in [(axes[0], 2, "size s"), (axes[1], 3, "duration T")]:
    for L in SIZES:
        rows = aval[(aval[:, 0] == L) & (aval[:, 1] >= WARMUP) & (aval[:, 4] == 0)]
        values = np.sort(rows[:, col])
        ccdf = 1.0 - np.arange(len(values)) / len(values)
        ax.loglog(values, ccdf, drawstyle="steps-post", color=colors[L], label=f"L = {L}")
    ax.set_xlabel(label)
    ax.set_ylabel(f"P({label.split()[-1]} ≥ x)")
    ax.legend()
axes[0].set_title("Avalanche sizes (complementary CDF)")
axes[1].set_title("Avalanche durations (complementary CDF)")
fig.tight_layout()
fig.savefig(FIGURES / "exp02_avalanches.png", dpi=150)
plt.show()

## 2c. Control: the same avalanches on a torus

$L$ = 64 torus, 2000 avalanches (first 500 as warm-up). Gliders cannot leave, so any glider released by an avalanche keeps circling.

In [ ]:
rng = np.random.default_rng(640)
state = settle(rng.random((64, 64)) < 0.35, CONWAY, TORUS, None, MAX_STEPS)
torus_records = []
start = time.perf_counter()
for i in range(2000):
    dead = np.argwhere(~state)
    cell = tuple(int(v) for v in dead[rng.integers(len(dead))])
    result = avalanche(state, cell, CONWAY, TORUS, None, MAX_STEPS)
    torus_records.append((i, result.size, result.duration, int(result.state is None)))
    if result.state is None:
        print(f"avalanche {i} hit the cap; stopping")
        break
    state = result.state
print(f"{len(torus_records)} torus avalanches in {time.perf_counter() - start:.1f} s")
torus = np.array(torus_records)
np.savetxt(
    DATA / "avalanches_torus.csv",
    torus,
    fmt="%d",
    delimiter=",",
    header="index,size,duration,censored",
    comments="",
)

rows = torus[(torus[:, 0] >= WARMUP) & (torus[:, 3] == 0)]
print(
    f"torus L = 64: median size {np.median(rows[:, 1]):.0f}, max size {rows[:, 1].max()}, "
    f"max duration {rows[:, 2].max()}"
)
describe(rows[:, 1], "size")
describe(rows[:, 2], "duration")
final_cycle = detect_cycle(state, CONWAY, TORUS, MAX_STEPS)
print(f"period of the final torus state: {final_cycle.period}")

## Results vs. predictions

*To be written after looking at the output above.*